# 02 — Normalize and QC: Engineering Scaffold

Phase-2 research semantics가 freeze되기 전의 notebook-first scaffold다. 실제 구현은 SSOT에 이미 고정된 NFC, BOM 제거, 외곽 trim뿐이며 나머지는 `BLOCKED_BY_P2_CONTRACT`로 유지한다.

## 0 Contract / Scope


In [ ]:
SYNTHETIC_ONLY = True
PHASE = '02'
BLOCKED_SEMANTICS = {
    'lid_algorithm', 'language_threshold', 'semantic_alignment_threshold',
    'accepted_rejected_policy', 'duplicate_survivor_semantics', 'manual_audit_cutoff',
}
assert SYNTHETIC_ONLY


## 1 Environment + Inputs


In [ ]:
import json
from tokenization_premium.paths import PROJECT_ROOT
from tokenization_premium.phase2 import (
    BLOCKED_BY_P2_CONTRACT, PAIR_REGISTRY_V001_RELATIVE_PATH,
    PAIR_REGISTRY_V002_RELATIVE_PATH, iter_parquet_batches, normalize_ssot_text,
    open_phase2_duckdb, write_parquet_batches_atomic,
)
from tokenization_premium.progress import ProgressHeartbeat, progress_tqdm

PAIR_REGISTRY_V001 = PROJECT_ROOT / PAIR_REGISTRY_V001_RELATIVE_PATH
PAIR_REGISTRY_V002 = PROJECT_ROOT / PAIR_REGISTRY_V002_RELATIVE_PATH
RUNTIME_DIR = PROJECT_ROOT / '.runtime/p2-duckdb-synthetic'
connection = open_phase2_duckdb(RUNTIME_DIR)
try:
    assert connection.execute('SELECT 1').fetchone() == (1,)
finally:
    connection.close()
RUNTIME_PREFLIGHT = 'SYNTHETIC_PASS'
RUNTIME_PREFLIGHT


## 2 D-01 Integrity Handoff


In [ ]:
D01_MANIFEST_PATH = PROJECT_ROOT / 'outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json'
d01_manifest = json.loads(D01_MANIFEST_PATH.read_text(encoding='utf-8'))
assert d01_manifest['validation']['status'] == 'PASS'
D01_ROW_COUNT = int(d01_manifest['pair_registry']['row_count'])
D01_HANDOFF = {
    'input': str(PAIR_REGISTRY_V001_RELATIVE_PATH),
    'prospective_output': str(PAIR_REGISTRY_V002_RELATIVE_PATH),
    'manifest_validation': d01_manifest['validation']['status'],
    'manifest_rows': D01_ROW_COUNT,
    'manifest_sha256': d01_manifest['pair_registry']['sha256'],
    'full_scan': 'NOT_RUN_SCAFFOLD',
}
PROSPECTIVE_LONG_STAGES = [
    {'phase': PHASE, 'stage': 'NORMALIZATION', 'total': D01_ROW_COUNT, 'checkpoint': 'NORMALIZATION_BATCH'},
    {'phase': PHASE, 'stage': 'QC_FLAG_COMPUTATION', 'total': D01_ROW_COUNT, 'checkpoint': 'QC_FLAG_BATCH'},
    {'phase': PHASE, 'stage': 'DUPLICATE_DISPOSITION', 'total': None, 'checkpoint': 'DUPLICATE_GROUP_BATCH'},
    {'phase': PHASE, 'stage': 'LID', 'total': D01_ROW_COUNT, 'checkpoint': 'LID_BATCH'},
    {'phase': PHASE, 'stage': 'SEMANTIC_QC', 'total': None, 'checkpoint': 'SEMANTIC_QC_BATCH'},
    {'phase': PHASE, 'stage': 'QC_FLOW', 'total': None, 'checkpoint': 'QC_FLOW_STAGE'},
    {'phase': PHASE, 'stage': 'REGISTRY_V002_WRITE', 'total': None, 'checkpoint': 'REGISTRY_V002_BATCH'},
    {'phase': PHASE, 'stage': 'ARTIFACT_HASH', 'total': None, 'checkpoint': 'ARTIFACT_HASH_CHUNK'},
]
D01_HANDOFF, PROSPECTIVE_LONG_STAGES


## 3 Normalization

SSOT-frozen operations only: NFC, BOM removal, outer trim. Internal whitespace is preserved.


In [ ]:
synthetic_inputs = ['e\u0301', '\ufeff  내부  공백  \ufeff', 'MiXeD ＡＢＣ']
normalization_results = []
with ProgressHeartbeat(
    run_id='p2-scaffold-synthetic', phase=PHASE, stage='NORMALIZATION_SYNTHETIC',
    total=len(synthetic_inputs),
) as heartbeat:
    for value in progress_tqdm(synthetic_inputs, desc='P2 synthetic normalization', position=1, leave=False):
        normalization_results.append(normalize_ssot_text(value))
        heartbeat.update(1)
    heartbeat.checkpoint('NORMALIZATION_SYNTHETIC_COMPLETE', normalized_rows=len(normalization_results))
normalization_results


## 4 QC Flag Computation


In [ ]:
QC_FLAG_COMPUTATION_STATUS = BLOCKED_BY_P2_CONTRACT
QC_FLAG_COMPUTATION_STATUS


## 5 Duplicate Disposition


In [ ]:
DUPLICATE_DISPOSITION_STATUS = BLOCKED_BY_P2_CONTRACT
DUPLICATE_DISPOSITION_STATUS


## 6 LID


In [ ]:
LID_STATUS = BLOCKED_BY_P2_CONTRACT
LID_STATUS


## 7 Semantic QC


In [ ]:
SEMANTIC_QC_STATUS = BLOCKED_BY_P2_CONTRACT
SEMANTIC_QC_STATUS


## 8 QC Flow


In [ ]:
QC_FLOW_STATUS = BLOCKED_BY_P2_CONTRACT
QC_FLOW_STATUS


## 9 Registry v002


In [ ]:
REGISTRY_V002_STATUS = BLOCKED_BY_P2_CONTRACT
STREAMING_INFRA = {
    'iterator': iter_parquet_batches.__name__,
    'atomic_writer': write_parquet_batches_atomic.__name__,
    'full_v002_generated': False,
}
REGISTRY_V002_STATUS, STREAMING_INFRA


## 10 Artifact / Hash


In [ ]:
ARTIFACT_HASH_STATUS = BLOCKED_BY_P2_CONTRACT
ATOMICITY_PATTERN = '*.partial -> validate -> os.replace'
FINAL_QC_ARTIFACT_GENERATED = False
ARTIFACT_HASH_STATUS, ATOMICITY_PATTERN, FINAL_QC_ARTIFACT_GENERATED


## 11 G1 Closure Evidence


In [ ]:
G1_CLOSURE_EVIDENCE_STATUS = BLOCKED_BY_P2_CONTRACT
SCAFFOLD_VERDICT = {
    'engineering_scaffold': 'READY',
    'research_contract': BLOCKED_BY_P2_CONTRACT,
    'g1_gate': 'OPEN',
}
G1_CLOSURE_EVIDENCE_STATUS, SCAFFOLD_VERDICT
